In [16]:
import jax
import jax_rmhd as jr
import jax.numpy as jnp
import jax.numpy.fft as ft
import matplotlib.pyplot as plt
import os
import jax_rmhd.snapshot_io as sn
from jax_rmhd.timestepping import rk_advance
jr.init_cluster()

#parameters
nx = 2048
ny = 2048
Lx = 2.0 * jnp.pi
Ly = 2.0 * jnp.pi
dx = Lx/nx
t = 0.0
t_snap=0.1
t_end=1.0
cfl_safety=1.0 #this is pretty aggressive
spatial_dimensions=2
snap_path="data/profiler/"

#we will use hyperviscosity
visc=1e-9
res=1e-9
hyper=3

x = jnp.linspace(0, Lx, nx, endpoint=False)
y = jnp.linspace(0, Ly, ny, endpoint=False)
x_grid = x.reshape(-1,1)
y_grid = y.reshape(1,-1)

#initialize arrays
#setting to zero to just test core code
phi = jnp.zeros_like(x_grid) + jnp.zeros_like(y_grid)
psi = jnp.zeros_like(phi)

#fft
phik=ft.rfft2(phi)
psik=ft.rfft2(psi)

#set up orbax snapshot manager
mngr=jr.snapshot_manager_setup(snap_path=snap_path,nsnap=1000)

#prepare necessary objects for simulation
params=jr.Parameters(nx=nx,ny=ny,Lx=Lx,Ly=Ly,visc=visc,res=res,hyper=hyper,cfl_safety=cfl_safety,dims=spatial_dimensions)
shardings=jr.setup_sharding(params)
kgrid = jr.setup_kgrids(params)
state = jr.SimulationState(t=0.0,fields=jr.Fields(phik,psik))

Running in local mode. Total devices: 1


In [ ]:
import time

_,_,state_sharding = shardings
rk_advance_jit=jax.jit(rk_advance,static_argnums=(2,),
                        in_shardings=(state_sharding, None),
                            out_shardings=state_sharding)

_ = rk_advance_jit(state,kgrid,params)
_ = rk_advance_jit(state,kgrid,params)

# 2. Time the Physics
nsteps=100
t0 = time.perf_counter()
for _ in range(nsteps):
    state = rk_advance_jit(state,kgrid,params)
state.fields.phik.block_until_ready() # CRITICAL for accurate timing
sim_time = (time.perf_counter() - t0) / nsteps

# 3. Time the I/O
nsnap=10
t1 = time.perf_counter()
for i in range(10):
    sn.save_snapshot(i,state,mngr)
io_time = (time.perf_counter() - t1)/nsnap

print(f"Physics Step: {sim_time:.4f}s")
print(f"Save Time:    {io_time:.4f}s")

---COMPILING rk_advance---
---COMPILING rk_advance---
Physics Step: 1.0798s
Save Time:    0.0223s


In [18]:
state_sharding

SimulationState(t=None, fields=Fields(phik=None, psik=None))